# 03 · Clusterização particional e hierárquica

**CRISP-DM fase 4** · K-Means, K-Medoids, hierárquica e a escolha de *k*

A pergunta desta etapa: **quantos grupos existem, e por que esse número e não outro?**

In [ ]:
import sys, warnings
from pathlib import Path

# O notebook roda a partir de `notebooks/`; a raiz do projeto entra no path
# para que `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from src import config
from src.utils.io import configurar_estilo

configurar_estilo()
print(f"Projeto em {RAIZ}")

In [ ]:
from src.data import dicionario as dic
from src.data.preparacao import carregar_curada, construir_matriz_modelagem, preparar_espaco
from src.model import distancias as dist
from src.model import estabilidade as est
from src.model import hierarquica as hier
from src.model import particional as part
from src.model import perfilagem as perf
from src.model import personas as pers

curada = carregar_curada()
matriz = construir_matriz_modelagem(curada, salvar=False)
euclidiano, nomes = preparar_espaco(matriz, "padrao")
euclidiano.shape

## 4.1 · A medida de distância

Três leituras do mesmo conjunto. A escolha não é detalhe de implementação: ela define
quem é vizinho de quem.

In [ ]:
dist.comparar_metricas(euclidiano, ("euclidean", "cityblock", "cosine"))

### A dissimilaridade de Gower

Para dados mistos. Cada variável contribui com um valor entre 0 e 1: diferença
normalizada pela amplitude, no caso das numéricas; 0 ou 1, no caso das nominais.

Com 2.925 imóveis a matriz ocupa cerca de 34 MB em `float32`. Acima de ~10.000
registros, amostre.

In [ ]:
gower = dist.gower(matriz, dic.NUMERICAS + dic.ORDINAIS, dic.NOMINAIS)
print(f"{gower.shape} · {gower.nbytes / 1e6:.1f} MB · faixa [{gower.min():.3f}, {gower.max():.3f}]")

## 4.2 · A escolha de *k*

Quatro métricas internas, mais o teste de estabilidade por reamostragem, mais o teto
operacional de cinco grupos. Nenhuma das métricas conhece o orçamento da diretoria —
o confronto entre elas e a viabilidade é a parte que importa.

In [ ]:
varredura = part.varrer_k(euclidiano, 2, 10)
varredura.round(4)

In [ ]:
print(f"Joelho da curva de inércia: k = {part.cotovelo(varredura['inercia'])}")
print(f"Maior silhueta: k = {varredura['silhueta'].idxmax()} ({varredura['silhueta'].max():.4f})")
print(f"Menor Davies-Bouldin: k = {varredura['davies_bouldin'].idxmin()}")

### O teste que derruba k = 5

Reamostrar 80% da carteira, reagrupar do zero e comparar com a partição de referência.
Se retirar um quinto dos imóveis muda a atribuição do restante, os grupos são artefato
do conjunto, não estrutura do mercado.

In [ ]:
estabilidade = est.estabilidade_por_k(euclidiano, range(2, 9), repeticoes=40)
estabilidade

Leitura: k = 2 e k = 4 se reproduzem (ARI acima de 0,90). **k = 5 não** — cai
para cerca de 0,44, que é pouco acima do que se esperaria de duas atribuições
aleatórias.

A decisão consolidada:

| Critério | Aponta para | Leitura |
|---|:-:|---|
| Cotovelo da inércia | 6 | mais grupos do que o negócio consegue operar |
| Maior silhueta global | 2 | "caro" e "barato" — a diretoria já sabe |
| Menor Davies-Bouldin dentro do teto | 5 | derrubado pela estabilidade |
| Estabilidade ARI ≥ 0,75 | 2 e 4 | os demais não se reproduzem |
| Restrição operacional | 2 a 5 | orçamento para cinco ações |
| **Decisão** | **4** | maior *k* estável, coeso acima do mínimo e viável |

In [ ]:
K = 4
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=K, init="k-means++", n_init=20, random_state=config.SEMENTE)
rotulos_kmeans = pers.ordenar_canonicamente(curada, kmeans.fit_predict(euclidiano))
np.bincount(rotulos_kmeans)

### A silhueta que a média esconde

A silhueta média global esconde o caso mais comum: um grupo coeso e outro esgarçado se
compensam e devolvem um número intermediário aceitável.

In [ ]:
part.silhueta_por_grupo(euclidiano, rotulos_kmeans)

## 4.3 · K-Medoids sobre Gower

A diferença que importa para o negócio: o centroide do K-Means é um imóvel médio que
**não existe**; o medoide é um imóvel **real da carteira**, com identificador.

In [ ]:
resultado = part.kmedoids(gower, K, reinicios=12)
rotulos_gower = pers.ordenar_canonicamente(curada, resultado.rotulos)
print(f"Custo: {resultado.custo:.1f} · iterações: {resultado.iteracoes}")
print(f"Tamanhos: {np.bincount(rotulos_gower)}")
print(f"Silhueta (no espaço de Gower): {part.metricas_internas(None, rotulos_gower, gower)['silhueta']:.4f}")

### Por que o resultado do Gower foi descartado

A silhueta do Gower é maior — mas está medida em **outro espaço métrico**, e comparar
as duas como se fossem a mesma escala é erro. O que decide é a leitura do resultado:

In [ ]:
perf.composicao_nominal(curada, rotulos_gower, "estilo")

Com peso igual por variável, `estilo` domina a partição: um grupo com 93% de
casas térreas, outro com 90% de sobrados. Isso é um **fato construtivo**, não uma
linha de produto — a Meridiano não tem política diferente para casa de um ou dois
pavimentos.

Hipótese derrubada, e registrada como tal.

## 4.4 · Clusterização hierárquica

Quatro critérios de ligação. Não é preciosismo: `single` produz encadeamento (um grupo
gigante que engole tudo), `complete` favorece grupos compactos, `average` fica no meio
e `ward` minimiza variância interna — por isso é o único que se comporta como o
K-Means.

In [ ]:
from scipy.spatial.distance import pdist

condensadas = pdist(euclidiano)
hier.comparar_ligacoes(euclidiano, condensadas, K)

O coeficiente cofenético favorece `average` (0,77), mas veja a coluna
`razao_maior_menor`: `average` e `single` produzem um grupo com 2.844 imóveis e outro
com **um único imóvel**. É o efeito de encadeamento, e nenhum orçamento se reparte
assim.

`ward` tem o pior cofenético e o melhor equilíbrio. É a escolha — e a justificativa
não é a métrica, é a viabilidade.

In [ ]:
ligacao = hier.matriz_ligacao(euclidiano, "ward")
hier.alturas_de_fusao(ligacao, ultimas=10)

In [ ]:
fig, eixo = plt.subplots(figsize=(11, 5), constrained_layout=True)
hier.desenhar_dendrograma(eixo, ligacao, K, "Ward — dendrograma truncado", config.CORAL)
plt.show()

## Comparação entre as três partições

Matriz de contingência e índice de Rand ajustado. É onde exatamente elas discordam.

In [ ]:
rotulos_ward = pers.ordenar_canonicamente(curada, hier.cortar(ligacao, K))

particoes = {
    "K-Means": rotulos_kmeans,
    "K-Medoids (Gower)": rotulos_gower,
    "Hierárquica (Ward)": rotulos_ward,
}
est.comparar_particoes(particoes)

In [ ]:
est.contingencia(rotulos_kmeans, rotulos_ward, "K-Means", "Ward")

## Perfilagem — do centroide à persona

In [ ]:
perf.perfil_por_segmento(curada, rotulos_kmeans).T

In [ ]:
assinatura = perf.assinatura(curada, rotulos_kmeans)
assinatura.T

In [ ]:
distintivos = perf.atributos_distintivos(assinatura, quantos=5)
for segmento, atributos in distintivos.items():
    nome = pers.PERSONAS[segmento]
    print(f"{nome['codigo']} — {nome['nome']}")
    for atributo, valor in atributos:
        print(f"    {atributo:<24} {valor:+.2f}")
    print()

### O representante real de cada segmento

A resposta para "me mostre um imóvel deste grupo". O centroide do K-Means não serve:
ele é uma média que pode ter 2,7 dormitórios.

In [ ]:
perf.medoides_reais(curada, rotulos_kmeans, gower)

In [ ]:
pers.tabela_personas()